# Sección 2.1 — Limpieza de las variables predictoras operativas

Parte del `dataset_trabajo.csv` (1.172 días válidos, generado en `correccion_fechas.ipynb`), donde los días inválidos ya fueron filtrados mirando el target `Frio (Kw)`. Esta sección limpia las **demás columnas predictoras**, que en el cierre de las 23:59 todavía tienen errores de medidor y nulos.

**Regla general para evitar fuga de información:** todo lo que se aprende de los datos (qué variable representa a cada grupo colineal, los umbrales de valores imposibles y las medianas para imputar) se calcula **solo con train** (hasta 2022-10-27, 820 días), usando el mismo corte cronológico 70/15/15 que `temporal_split()` en `src/preprocessing_pipeline.py`. Validación, test y los datos nuevos de `predict.py` se transforman con esos valores, sin recalcular nada.

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.auxiliar_functions import PROCESSED_DATA_DIR

COL_TARGET = "Frio (Kw)_Consolidado EE"
df_horario = pd.read_csv(PROCESSED_DATA_DIR / "dataset_trabajo.csv", parse_dates=["DIA"], low_memory=False)

# Cierre diario (23:59) = total del día de cada totalizador
df_diario = (df_horario[df_horario["HORA"] == "23:59"]
             .drop(columns="HORA").sort_values("DIA").reset_index(drop=True).copy())

# Target D+1 por fecha calendario (si falta el día siguiente, queda NaN)
frio_por_dia = df_diario.set_index("DIA")[COL_TARGET]
df_diario["y"] = (df_diario["DIA"] + pd.Timedelta(days=1)).map(frio_por_dia)

# Mismo corte cronológico 70/15/15 que temporal_split() de src/preprocessing_pipeline.py
n = len(df_diario)
n_train, n_val = round(n * 0.70), round(n * 0.15)
FIN_TRAIN = df_diario["DIA"].iloc[n_train - 1]
FIN_VAL = df_diario["DIA"].iloc[n_train + n_val - 1]
es_train = df_diario["DIA"] <= FIN_TRAIN

print(f"Días: {n} ({df_diario['DIA'].min().date()} a {df_diario['DIA'].max().date()}) | columnas: {df_diario.shape[1]}")
print(f"Train hasta {FIN_TRAIN.date()} ({es_train.sum()} días) | Val hasta {FIN_VAL.date()} | Test hasta {df_diario['DIA'].max().date()}")

Días: 1172 (2020-07-01 a 2023-10-25) | columnas: 421
Train hasta 2022-10-27 (820 días) | Val hasta 2023-04-28 | Test hasta 2023-10-25


In [3]:
CANDIDATAS = [
    "Hl Producido Bodega_Consolidado Produccion", "Hl Cerveza Envasada_Consolidado Produccion",
    "Hl Cerveza Filtrada_Consolidado Produccion", "Hl de Mosto_Consolidado Produccion",
    "Hl Cerveza Cocina_Consolidado Produccion", "Hl Cerveza L4_Consolidado Produccion",
    "Agua Planta (Hl)_Consolidado Agua", "Planta de agua (Hl)_Consolidado Agua",
    "Produccion (Hl)_Consolidado Agua", "Agua Cond Evaporativos_Consolidado Agua",
    "Agua Envasado (Hl)_Consolidado Agua", "Agua Elaboracion (Hl)_Consolidado Agua",
    "Vapor Envasado (Kg)_Consolidado GasVapor", "Tot_Vapor_L3_L4_Consolidado GasVapor",
    "Elaboracion (Kw)_Consolidado EE", "Aire (Kw)_Consolidado EE", "Bodega (Kw)_Consolidado EE",
    "Envasado (Kw)_Consolidado EE",
    "KW Cond 5. 6 y 9_Totalizadores Energia", "KW Trafo 10_Totalizadores Energia",
    "KW Trafo 11_Totalizadores Energia", "KW Trafo 9_Totalizadores Energia",
    "Kw Compresores Aire_Totalizadores Energia",
    "Aire Producido (M3)_Consolidado Aire", "Aire Envasado (M3)_Consolidado Aire",
    "Aire Planta (M3)_Consolidado Aire",
    "Totalizador_Filtracion_Totalizadores CO2", "Totalizador_Tq_Pulmon_Totalizadores CO2",
]
# Proxies del target (Sección 1.3): contienen el Frío del mismo día -> no son candidatas
PROXIES = ["Sala Maq (Kw)_Consolidado EE", "Servicios (Kw)_Consolidado EE", "Planta (Kw)_Consolidado EE",
           "KW Gral Planta_Totalizadores Energia"] + [f"KW Mycom {i}_Totalizadores Energia" for i in range(1, 8)]
assert not set(CANDIDATAS) & set(PROXIES), "Hay un proxy del target entre las candidatas"

MAX_NULOS_TRAIN = 0.20   # más de 20 % de nulos en train: no hay base suficiente para imputar
UMBRAL_COLINEAL = 0.90   # |Spearman| a partir del cual dos predictoras son redundantes


def seleccionar_representantes(df, columnas, col_y="y", max_nulos=MAX_NULOS_TRAIN, umbral=UMBRAL_COLINEAL):
    """Elige una variable por grupo colineal usando SOLO el DataFrame que recibe (train).

    1. Descarta columnas con más de `max_nulos` de nulos.
    2. Ordena las restantes por |Spearman| con el target, de mayor a menor.
    3. Conserva una variable solo si su |Spearman| con todas las ya conservadas es menor
       a `umbral`; si no, es redundante con una que se relaciona mejor con el target.
    """
    nulos = df[columnas].isna().mean()
    rho_y = df[columnas].corrwith(df[col_y], method="spearman")
    corr = df[columnas].corr(method="spearman").abs()

    decision, elegidas = {}, []
    for col in rho_y.abs().sort_values(ascending=False).index:
        if nulos[col] > max_nulos:
            decision[col] = f"descartada: {nulos[col]:.0%} de nulos en train"
            continue
        redundante_con = [e for e in elegidas if corr.loc[col, e] >= umbral]
        if redundante_con:
            decision[col] = f"redundante con {redundante_con[0]} (|rho| = {corr.loc[col, redundante_con[0]]:.3f})"
        else:
            elegidas.append(col)
            decision[col] = "elegida"

    tabla = pd.DataFrame({"spearman_y_train": rho_y.round(3), "nulos_train": nulos.round(3),
                          "decision": pd.Series(decision)}).sort_values("spearman_y_train", key=abs, ascending=False)
    return elegidas, tabla


PREDICTORAS, tabla_seleccion = seleccionar_representantes(df_diario[es_train], CANDIDATAS)
print(f"Predictoras elegidas: {len(PREDICTORAS)} de {len(CANDIDATAS)} candidatas")
tabla_seleccion

Predictoras elegidas: 20 de 28 candidatas


,spearman_y_train,nulos_train,decision
KW Trafo 11_Totalizadores Energia,0.708,0.000,elegida
Agua Planta (Hl)_Consolidado Agua,0.688,0.000,elegida
Planta de agua (Hl)_Consolidado Agua,0.675,0.000,redundante con Agua Planta (Hl)_Consolidado Ag...
KW Cond 5. 6 y 9_Totalizadores Energia,0.673,0.000,elegida
Produccion (Hl)_Consolidado Agua,0.635,0.000,redundante con Agua Planta (Hl)_Consolidado Ag...
KW Trafo 9_Totalizadores Energia,0.590,0.000,elegida
KW Trafo 10_Totalizadores Energia,0.555,0.000,elegida
Aire Producido (M3)_Consolidado Aire,0.544,0.000,elegida
Kw Compresores Aire_Totalizadores Energia,0.542,0.000,redundante con Aire Producido (M3)_Consolidado...
Agua Envasado (Hl)_Consolidado Agua,0.530,0.000,elegida


**Selección de un representante por grupo colineal.** Se parte de las 28 candidatas de la Sección 1.3 y se excluyen los proxies del target (`Sala Maq`, `Servicios`, `Planta`, `KW Gral Planta` y los compresores `KW Mycom`). La función `seleccionar_representantes` ordena las candidatas por |Spearman| con el target en train y conserva una variable solo si su |Spearman| con todas las ya elegidas es menor a 0,9. Se descartan además las columnas con más de 20 % de nulos en train.

Resultado: **20 predictoras de 28**. Los grupos redundantes se resolvieron así:
- Agua total: se conserva `Agua Planta (Hl)`; salen `Planta de agua (Hl)` (0,95) y `Produccion (Hl)` (0,91).
- Aire comprimido: se conserva `Aire Producido (M3)`; salen `Kw Compresores Aire` (0,99) y `Aire (Kw)` (0,98).
- Envasado: se conserva `Envasado (Kw)`; sale `Hl Cerveza Envasada` (0,92).
- Cocina: se conserva `Hl Cerveza Cocina`; sale `Hl de Mosto` (0,95).
- Vapor: se conserva `Vapor Envasado (Kg)`; sale `Tot_Vapor_L3_L4` (0,90).
- `Agua Cond Evaporativos` se descarta por nulos: no existe en el archivo 2020_2022, lo que en train representa un 43 % de días sin dato.

In [4]:
from sklearn.pipeline import Pipeline
from src.limpieza_predictoras import LimpiadorExtremos, ImputadorTemporal

limpieza_predictoras = Pipeline([
    ("extremos", LimpiadorExtremos(factor=5.0, cuantil=0.99)),
    ("imputacion", ImputadorTemporal(limite=3)),
])

# fit SOLO con train: los topes (5 x P99) y las medianas salen de train
limpieza_predictoras.fit(df_diario.loc[es_train, PREDICTORAS])

# transform sobre todo el período: val y test usan los valores aprendidos en train
X_limpio = limpieza_predictoras.transform(df_diario[PREDICTORAS])

extremos = limpieza_predictoras.named_steps["extremos"]
imputacion = limpieza_predictoras.named_steps["imputacion"]
resumen_21 = pd.DataFrame({
    "tope_5xP99_train": extremos.topes_.round(0),
    "nulos_originales": df_diario[PREDICTORAS].isna().sum(),
    "imposibles_a_NaN": extremos.reemplazos_,
    "imputados_ffill": imputacion.imputados_ffill_,
    "imputados_mediana_train": imputacion.imputados_mediana_,
})
print("Columnas con algún cambio:")
print(resumen_21[resumen_21.iloc[:, 1:].sum(axis=1) > 0].to_string())
print(f"\nNaN restantes: {int(X_limpio.isna().sum().sum())} | X_limpio: {X_limpio.shape}")

# Qué valores se consideraron imposibles y en qué partición caen
particion = np.where(es_train, "train", np.where(df_diario["DIA"] <= FIN_VAL, "val", "test"))
es_imposible = (df_diario[PREDICTORAS] < 0) | (df_diario[PREDICTORAS] > extremos.topes_)
print("\nImposibles por partición:", es_imposible.groupby(particion).sum().sum(axis=1).to_dict())
for col in PREDICTORAS:
    if es_imposible[col].any():
        print(f"  {col}: {df_diario.loc[es_imposible[col], ['DIA', col]].values.tolist()}")

Columnas con algún cambio:
                                          tope_5xP99_train  nulos_originales  imposibles_a_NaN  imputados_ffill  imputados_mediana_train
Elaboracion (Kw)_Consolidado EE                    54049.0                 0                 1                1                        0
Envasado (Kw)_Consolidado EE                      107225.0                 0                 1                1                        0
Bodega (Kw)_Consolidado EE                         47656.0                 0                 6                6                        0
Totalizador_Tq_Pulmon_Totalizadores CO2           100804.0                 1                 1                2                        0
Totalizador_Filtracion_Totalizadores CO2           34908.0                 1                 0                1                        0

NaN restantes: 0 | X_limpio: (1172, 20)

Imposibles por partición: {'test': 0, 'train': 7, 'val': 2}
  Elaboracion (Kw)_Consolidado EE: [[Timestamp('2

**Limpieza de valores imposibles e imputación** (`src/limpieza_predictoras.py`), como transformadores de sklearn para reutilizarlos en el Pipeline y en `predict.py`:

1. `LimpiadorExtremos` pasa a `NaN` los valores **negativos** (regla física: un totalizador diario no puede ser negativo, independiente de los datos) y los mayores a **5 × P99 de train** (umbral estadístico, aprendido en `fit` solo con train).
2. `ImputadorTemporal` completa los `NaN` con **forward fill limitado a 3 días** (copia el último cierre anterior, nunca uno posterior) y lo que quede con la **mediana de train**.

Resultado: **9 valores imposibles** (7 en train, 2 en validación, 0 en test) y 2 nulos originales, todos completados con forward fill; quedan 0 nulos. Los casos detectados:
- `Bodega (Kw)`: 6 cierres negativos (entre −236 y −6.247 kW).
- `Envasado (Kw)`: 4.504.000 kW el 2021-11-29 (su tope en train es 107.225 kW).
- `Elaboracion (Kw)`: −672 kW el 2021-11-02.
- `Totalizador_Tq_Pulmon`: −805.849 el 2020-08-28. Este valor explica la diferencia entre su Pearson y su Spearman observada en la exploración de *Totalizadores CO2*.

La limpieza no escala: los valores quedan en sus unidades originales, de acuerdo con la consigna para `dataset_final.csv`. El escalado se ajusta con `X_train` en `build_scaler_pipeline()`.